In [6]:
import numpy as np
import pandas as pd
TARGETS = ["X"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [7]:
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        for col in [col_r2, col_mse]:
            if col in results.columns:
                results[col] = (
                    results[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )

In [8]:
best_models_tables = {}
best_only = []

summary_all = []     # estatísticas com todos
summary_topN = []   # estatísticas com top 10

N = 10

import pandas as pd

def summarize_models(results, TARGETS, SETS, N=10, output="Resumo_Estatisticas.xlsx"):
    
    best_models_tables = {}
    best_only = []
    summary_all = []
    summary_top10 = []

    for target in TARGETS:
        for dataset in SETS:

            col_r2 = f"R2_{dataset}_{target}"
            col_mse = f"MSE_{dataset}_{target}"

            if col_r2 not in results.columns:
                continue

            table = results.copy()

            table = table.sort_values(by=col_r2, ascending=False)

            best_models_tables[f"{dataset}_{target}"] = table

            # 🔹 melhor modelo
            best = table.iloc[0]

            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": best["model"],
                "Neurons": best["Neurons"],
                "Best_R2": best[col_r2]
            })

            # 🔹 estatísticas de todos
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })

            # 🔹 estatísticas top N
            topN = table.head(N)

            summary_topN.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": topN[col_r2].mean(),
                "std_r2": topN[col_r2].std(),
                "mean_mse": topN[col_mse].mean(),
                "std_mse": topN[col_mse].std()
            })

    df_summary_all = pd.DataFrame(summary_all)
    df_summary_topN = pd.DataFrame(summary_topN)
    best_only_df = pd.DataFrame(best_only)
    # exportar
    with pd.ExcelWriter(output) as writer:
        df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
        df_summary_topN.to_excel(writer, sheet_name="Top_Modelos", index=False)
        best_only_df.to_excel(writer, sheet_name="Melhor_Modelo", index=False)

    return best_models_tables, df_summary_all, df_summary_topN, best_only_df

In [9]:
tables, summary_all, summary_top10, best_models = summarize_models(
    results, TARGETS, SETS
)

In [10]:
summary_top10["bestR2"] = best_models["Best_R2"]
summary_top10["Neurons"] = best_models["Neurons"]
display(summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons
0,Train_1,X,0.872424,0.038004,0.013756,0.004098,0.928231,[264]
1,Train_2,X,-0.606523,0.821139,0.135366,0.069189,0.541638,"[264, 128]"
2,Val,X,0.949007,0.003710,0.004876,0.000355,0.954110,[264]
3,Test_1,X,0.904099,0.033506,0.010047,0.003510,0.953086,[264]
4,Test_2,X,-0.570426,0.763393,0.129235,0.062822,0.481260,"[264, 128]"
5,Test_3,X,0.979480,0.002363,0.001888,0.000217,0.982142,"[264, 128, 64]"


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted